# H12 · Balance sheets: do firms in more developed countries run more debt?

> **Hypothesis H12:** Non-financial companies in more developed countries finance a smaller share of their assets with equity (they carry more debt).

| | |
|---|---|
| **Why we ask** | ROA is only one profitability measure. How a company is financed is the other side of the same coin: deep credit markets and strong creditor protection let firms borrow, while firms in weaker systems rely on retained earnings. Notebook 1.3 found that leverage did not explain the ROA link; here financing is the outcome in its own right. |
| **Prediction** | **Negative** association between the development index and the equity ratio (equity / total assets). |
| **Data** | Yahoo company-years of **non-financial** companies (banks and insurers have a different balance-sheet logic). Equity ratio in percent, winsorised at 1% / 99%. Development index of the previous year. |
| **Primary test (fixed before looking at the result)** | `equity ratio (pp) ~ development index + sector + size rank + fiscal year`, errors clustered by country, wild-bootstrap p-value. |
| **Verdict rule** | *Supported* if the effect has the predicted sign, the wild-bootstrap p < 0.05, and the sign holds in at least 80% of the robustness samples. *Partly supported* if it has the predicted sign and p < 0.10 (or p < 0.05 but robustness is below 80%). *Contradicted* if significantly the opposite way. Otherwise *not supported*. |

**Honest framing.** Several hypotheses in this series (H1 to H3) were suggested by an exploratory scan of the same data, so their p-values are optimistic and a pass is a lead, not a confirmation. Where possible the result is **replicated on ESEF**, an independent source (which still shares some companies with Yahoo). All hypotheses are corrected for multiple testing together in `1.0_hypotheses.ipynb`. Everything is **association, not causation**.

In [1]:
import sys, pathlib, warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
import statsmodels.formula.api as smf
import patsy

warnings.filterwarnings("ignore")
ROOT = pathlib.Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src" / "analysis"))
import prep, utils
from utils import CONTROLS, fit_cluster, wild_cluster, spearman, boot_ci, partial_rho, bh, scan

con = prep.connect()
q = lambda sql: con.sql(sql).df()
pd.set_option("display.width", 220, "display.max_columns", 40, "display.max_rows", 120, "display.float_format", "{:,.3f}".format)
px.defaults.template = "plotly_white"

country = prep.load_country(con)
dim, profile, profile_all, panel = country["dim"], country["profile"], country["profile_all"], country["panel"]
feat_cols = country["feat_cols"]
fy = prep.load_firms(con, country)               # Yahoo company-years (primary sample)
names = dim.name
MIN_FIRMS = prep.MIN_FIRMS
print(f"{len(dim)} countries | {len(feat_cols)} country features | {len(fy):,} Yahoo company-years of {fy.company_id.nunique():,} companies")

SAMPLES = {
 "all firms": lambda d: d,
 "excluding financials & real estate": lambda d: d[~d.sector.isin(["Financial Services", "Real Estate"])],
 "excluding small-sample countries": lambda d: d[~d.iso3.isin(dim.index[dim.small_firm_sample])],
 "excluding USA, China, Japan": lambda d: d[~d.iso3.isin(["USA", "CHN", "JPN"])],
 "fiscal years 2023 and later": lambda d: d[d.fiscal_year >= 2023],
 "EU countries only": lambda d: d[d.iso3.isin(dim.index[dim.is_eu])],
 "non-EU countries only": lambda d: d[~d.iso3.isin(dim.index[dim.is_eu])],
}
def robustness(focal, outcome, expected_sign, B=499, extra="", controls=CONTROLS, data=None):
    """Re-estimate the primary test in each sample. Returns a table and the share of samples with the expected sign."""
    rows = []
    for name, flt in SAMPLES.items():
        r = wild_cluster(focal, flt(fy if data is None else data), outcome=outcome, extra=extra, B=B, controls=controls)
        rows.append({"sample": name, "coef": r["coef"], "ci_low": r["ci_low"], "ci_high": r["ci_high"], "p (wild)": r["p_wild"],
                     "countries": r["clusters"], "firm-years": r["n"]})
    t = pd.DataFrame(rows).set_index("sample")
    same = (np.sign(t.coef) == expected_sign) if expected_sign else pd.Series(True, index=t.index)
    return t.assign(**{"expected sign": same}), float(same.mean())

34 countries | 35 country features | 10,220 Yahoo company-years of 2,590 companies


## 1 · The balance-sheet and profit measures
Five more outcomes describe how a company is financed and how it earns: equity ratio, asset turnover, net margin, operating margin and return on equity. Loss-making firm-years are included as a share.

In [2]:
nf = fy[~fy.is_financial].copy()
nf["equity_ratio_pp"] = nf.equity_ratio_w * 100
nf["turnover_x"] = nf.turnover_w
nf["net_margin_pp"] = nf.net_margin_w * 100
nf["op_margin_pp"] = nf.operating_margin_w * 100
nf["roe"] = nf.net_income / nf.equity.where(nf.equity > 0); lo, hi = nf.roe.quantile(prep.WINSOR); nf["roe_pp"] = nf.roe.clip(lo, hi) * 100
OUT = {"equity ratio (pp of assets)": "equity_ratio_pp", "asset turnover (revenue / assets)": "turnover_x", "net margin (pp)": "net_margin_pp",
       "operating margin (pp)": "op_margin_pp", "return on equity (pp)": "roe_pp", "loss-making firm-years (%)": "loss_pp"}
cm = nf.groupby("iso3")[list(OUT.values())].median().join(nf.groupby("iso3").company_id.nunique().rename("firms")).query("firms >= @MIN_FIRMS").join(country["dev_idx"].rename("dev"))
fig = make_subplots(rows=2, cols=3, subplot_titles=[f"{k} (ρ = {spearman(cm.dev, cm[v])[0]:+.2f})" for k, v in OUT.items()], vertical_spacing=0.16)
for i, (k, v) in enumerate(OUT.items()):
    r_, c_ = divmod(i, 3); d = cm[["dev", v]].dropna()
    fig.add_trace(go.Scatter(x=d.dev, y=d[v], mode="markers+text", text=d.index, textposition="top center", textfont_size=7, marker=dict(size=6, color="#4C78A8"), showlegend=False), row=r_ + 1, col=c_ + 1)
    b, a = np.polyfit(d.dev, d[v], 1); xs = np.array([d.dev.min(), d.dev.max()])
    fig.add_trace(go.Scatter(x=xs, y=a + b * xs, mode="lines", line=dict(color="#E45756"), showlegend=False), row=r_ + 1, col=c_ + 1)
fig.update_layout(height=700, title="Country medians (non-financial firms) vs development index"); fig.show()
print(f"{len(nf):,} non-financial company-years, {nf.company_id.nunique():,} companies.")
nf.groupby("sector")[["equity_ratio_pp", "net_margin_pp", "roe_pp"]].median().round(1)

9,051 non-financial company-years, 2,293 companies.


,equity_ratio_pp,net_margin_pp,roe_pp
sector,,,
Basic Materials,49.600,6.600,9.300
Communication Services,41.400,8.100,9.500
Consumer Cyclical,40.100,5.700,12.300
Consumer Defensive,45.600,5.400,11.400
Energy,42.400,7.500,11.900
Healthcare,51.700,7.200,9.700
Industrials,37.500,6.000,13.000
Real Estate,48.100,23.200,6.500
Technology,53.500,7.600,12.800


## 2 · Primary test

In [3]:
primary = wild_cluster("dev_z", nf, outcome="equity_ratio_pp", B=1999)
print(f"Typical equity ratio: {nf.equity_ratio_pp.median():.0f}% of assets (median).")
print(f"+1 SD development → {primary['coef']:+.2f} pp equity ratio (95% CI {primary['ci_low']:+.2f} to {primary['ci_high']:+.2f}).")
print(f"p (cluster-robust) = {primary['p_cluster']:.4f} | p (wild bootstrap) = {primary['p_wild']:.4f} | {primary['clusters']} countries, {primary['n']:,} company-years")

Typical equity ratio: 44% of assets (median).
+1 SD development → -0.29 pp equity ratio (95% CI -1.72 to +1.14).
p (cluster-robust) = 0.6937 | p (wild bootstrap) = 0.7020 | 34 countries, 9,051 company-years


### All outcomes together
The same model for each outcome; Benjamini–Hochberg corrected across the six. Loss share is a linear probability model (the share of firm-years in percentage points).

In [4]:
rows = []
for lab, out in OUT.items():
    r = wild_cluster("dev_z", nf, outcome=out, B=999)
    rows.append({"outcome": lab, "coef per +1 SD": r["coef"], "ci_low": r["ci_low"], "ci_high": r["ci_high"], "p (wild)": r["p_wild"], "median": nf[out].median(), "n": r["n"]})
secondary = pd.DataFrame(rows).set_index("outcome"); secondary["q (BH)"] = bh(secondary["p (wild)"])
secondary["effect as % of median"] = secondary["coef per +1 SD"] / secondary["median"].abs() * 100
secondary

,coef per +1 SD,ci_low,ci_high,p (wild),median,n,q (BH),effect as % of median
outcome,,,,,,,,
equity ratio (pp of assets),-0.290,-1.721,1.141,0.721,44.241,9051,0.747,-0.655
asset turnover (revenue / assets),-0.012,-0.056,0.032,0.747,0.603,9037,0.747,-1.967
net margin (pp),-3.637,-5.401,-1.872,0.001,7.073,8932,0.004,-51.417
operating margin (pp),-3.551,-5.450,-1.652,0.005,11.475,8911,0.008,-30.948
return on equity (pp),-2.407,-3.621,-1.193,0.002,10.329,8865,0.004,-23.302
loss-making firm-years (%),3.401,1.434,5.368,0.002,0.000,9051,0.004,inf


## 3 · How ROE splits into financing and profit
Return on equity = ROA × (assets / equity). If more developed countries have both lower ROA and more leverage, the two partly offset in ROE. The table gives each component's response to development at the country-median level.

In [5]:
cm["leverage (assets/equity, median)"] = 100 / cm.equity_ratio_pp
cmp_ = pd.DataFrame({"outcome": ["ROA (from H0)", "leverage (assets / equity)", "ROE"],
                      "ρ with development": [spearman(cm.dev, cm.join(nf.groupby("iso3").roa_pp.median().rename("roa")).roa)[0], spearman(cm.dev, cm["leverage (assets/equity, median)"])[0], spearman(cm.dev, cm.roe_pp)[0]]}).set_index("outcome")
cmp_

,ρ with development
outcome,
ROA (from H0),-0.429
leverage (assets / equity),-0.166
ROE,-0.493


## 4 · Robustness
The primary test in seven samples (wild bootstrap, 499 draws each).

In [6]:
rob, rob_share = robustness("dev_z", "equity_ratio_pp", -1, data=nf)
fig = px.scatter(rob.reset_index(), x="coef", y="sample", error_x=rob.ci_high - rob.coef, error_x_minus=rob.coef - rob.ci_low, title="Development → equity ratio (pp per +1 SD) in different samples")
fig.add_vline(x=0, line_color="grey"); fig.update_layout(height=420, yaxis_autorange="reversed"); fig.show()
print(f"Predicted sign (negative) in {rob_share:.0%} of samples.")
rob

Predicted sign (negative) in 100% of samples.


,coef,ci_low,ci_high,p (wild),countries,firm-years,expected sign
sample,,,,,,,
all firms,-0.290,-1.721,1.141,0.704,34,9051,True
excluding financials & real estate,-0.060,-1.718,1.597,0.936,34,8063,True
excluding small-sample countries,-0.337,-1.810,1.136,0.692,29,8686,True
"excluding USA, China, Japan",-0.231,-1.707,1.244,0.782,31,7966,True
fiscal years 2023 and later,-0.532,-2.068,1.004,0.504,34,6737,True
EU countries only,-0.070,-4.570,4.429,0.978,21,4493,True
non-EU countries only,-0.180,-2.186,1.827,0.876,13,4558,True


## 5 · Replication on ESEF
EU companies, no sector control and no clean way to separate financials, so all ESEF companies are used.

In [7]:
fe = prep.load_firms(con, country, source="ESEF"); fe["equity_ratio_pp"] = fe.equity_ratio_w * 100
rep = wild_cluster("dev_z", fe, outcome="equity_ratio_pp", controls="size_rank + C(fiscal_year)", B=1999)
rep_same = bool(np.sign(rep["coef"]) == -1)
print(f"ESEF ({rep['n']:,} company-years, {rep['clusters']} countries): {rep['coef']:+.2f} pp (95% CI {rep['ci_low']:+.2f} to {rep['ci_high']:+.2f}), wild p = {rep['p_wild']:.3f}")

ESEF (5,247 company-years, 19 countries): +0.69 pp (95% CI -4.90 to +6.28), wild p = 0.838


## 6 · Verdict

In [8]:
v = utils.verdict(primary["coef"], primary["p_wild"], -1, rob_share, primary["clusters"])
sg = secondary[secondary["q (BH)"] < 0.10]
lines = [
 f"Primary: +1 SD development → {primary['coef']:+.2f} pp equity ratio (typical {nf.equity_ratio_pp.median():.0f}%), wild p = {primary['p_wild']:.3f}, {primary['clusters']} countries, {primary['n']:,} company-years.",
 f"Robustness: predicted sign in {rob_share:.0%} of seven samples. ESEF replication: {rep['coef']:+.2f} pp (wild p = {rep['p_wild']:.3f}), " + ("same direction." if rep_same else "opposite direction."),
 "Outcomes with q < 0.10 among the six: " + (", ".join(f"{i} ({r['coef per +1 SD']:+.2f})" for i, r in sg.iterrows()) or "none") + ".",
 f"VERDICT H12: {v.upper()}",
]
print("\n".join(lines))
utils.save_result("H12", id="H12", title="Financing: equity ratio and development", statement="Non-financial firms in more developed countries have a lower equity ratio.",
                  expected_sign=-1, effect=primary["coef"], ci_low=primary["ci_low"], ci_high=primary["ci_high"], effect_unit="pp equity ratio per +1 SD development",
                  p_primary=primary["p_wild"], p_method="wild cluster bootstrap", n=primary["n"], clusters=primary["clusters"], robust_share=rob_share,
                  replication_coef=rep["coef"], replication_p=rep["p_wild"], replication_same_sign=rep_same, verdict=v,
                  notebook="1.12_balance_sheets_and_financing.ipynb", notes=lines)

Primary: +1 SD development → -0.29 pp equity ratio (typical 44%), wild p = 0.702, 34 countries, 9,051 company-years.
Robustness: predicted sign in 100% of seven samples. ESEF replication: +0.69 pp (wild p = 0.838), opposite direction.
Outcomes with q < 0.10 among the six: net margin (pp) (-3.64), operating margin (pp) (-3.55), return on equity (pp) (-2.41), loss-making firm-years (%) (+3.40).
VERDICT H12: NOT SUPPORTED


### Limits
- Book equity is an accounting number: buybacks, goodwill write-offs and revaluations can move it independently of actual financing choices.
- Only companies with two or more of the needed statement items are included, and listed firms are not typical of how most firms in a country finance themselves.
- Leases and pension liabilities are treated differently across accounting standards (IFRS, US GAAP, local rules), which affects total assets and equity.
- Association, not causation.

In [9]:
con.close()